# Atividade Prática: O Problema da Soma Máxima de Subvetor

**Disciplina:** Algoritmos e Estruturas de Dados (2026)  
**Tema:** Comparação de abordagens algorítmicas para o problema da soma máxima de subvetor  
**Implementação:** Python + Cython

Este notebook implementa, em células separadas, os **5 algoritmos** discutidos na atividade e no artigo-base de Jon Bentley:

1. Algoritmo 1 — abordagem cúbica, usada como base de comparação.
2. Algoritmo 2a — abordagem quadrática por reaproveitamento da soma corrente.
3. Algoritmo 2b — abordagem quadrática com vetor de somas acumuladas.
4. Algoritmo 3 — abordagem por divisão e conquista.
5. Algoritmo 4 — abordagem linear, também conhecida como algoritmo de Kadane.

A definição adotada segue o artigo: se todos os valores forem negativos, a resposta é `0`, pois considera-se permitido escolher o subvetor vazio.


## 1. Preparação do ambiente

As células abaixo carregam o Cython no notebook e importam bibliotecas usadas para testes e medições.


In [2]:
%load_ext Cython

import numpy as np
import pandas as pd
import timeit
import matplotlib.pyplot as plt

The Cython extension is already loaded. To reload it, use:
  %reload_ext Cython


## 2. Exemplo-base do artigo

O artigo utiliza o vetor:

`[31, -41, 59, 26, -53, 58, 97, -93, -23, 84]`

A maior soma aparece no subvetor `[59, 26, -53, 58, 97]`, cujo resultado é `187`.


In [4]:
x_exemplo = np.array([31, -41, 59, 26, -53, 58, 97, -93, -23, 84], dtype=np.float64)

print("Vetor:", x_exemplo)
print("Resultado esperado:", 187.0)


Vetor: [ 31. -41.  59.  26. -53.  58.  97. -93. -23.  84.]
Resultado esperado: 187.0


# Algoritmo 1 — Solução Cúbica `O(n³)`

## Ideia

A solução cúbica testa todos os pares possíveis de início e fim do subvetor. Para cada par `(L, U)`, ela calcula a soma de `x[L:U+1]` percorrendo novamente o intervalo.

Em outras palavras:

1. Escolhe um início `L`;
2. escolhe um fim `U`;
3. soma todos os elementos entre `L` e `U`;
4. compara essa soma com a melhor soma encontrada até o momento.

## Melhoria em relação a nada

Este algoritmo não é uma otimização. Ele é a **base ingênua** da atividade. Sua importância está em mostrar por que repetir recomputações torna o programa inviável para entradas grandes.

## Linha crítica

A linha crítica é:

```python
soma += x[i]
```

dentro do terceiro laço. Essa linha é executada para cada elemento de cada subvetor possível. Como existem `O(n²)` subvectores e cada um pode custar até `O(n)` para ser somado, o custo total é `O(n³)`.

## Análise assintótica detalhada

Para cada valor de `L`, o algoritmo percorre todos os valores de `U >= L`. Isso gera aproximadamente:

\[
\frac{n(n+1)}{2}
\]

pares `(L, U)`, ou seja, `O(n²)` subvectores.

Para cada par, a soma do subvetor pode visitar até `n` elementos. Logo:

\[
O(n^2) \cdot O(n) = O(n^3)
\]

### Complexidade

- Tempo: `O(n³)`
- Espaço adicional: `O(1)`


In [5]:
%%cython

cpdef double max_subarray_cubic(double[:] x):
    cdef Py_ssize_t n = x.shape[0]
    cdef Py_ssize_t L, U, i
    cdef double max_so_far = 0.0
    cdef double soma

    for L in range(n):
        for U in range(L, n):
            soma = 0.0
            for i in range(L, U + 1):
                soma += x[i]
            if soma > max_so_far:
                max_so_far = soma

    return max_so_far


Content of stdout:
_cython_magic_88a3b3b31dd4194a8ffd54d6823ccab72282f2e534bc59430d72437fb19c0afd.c
   Criando biblioteca C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_88a3b3b31dd4194a8ffd54d6823ccab72282f2e534bc59430d72437fb19c0afd.cp312-win_amd64.lib e objeto C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_88a3b3b31dd4194a8ffd54d6823ccab72282f2e534bc59430d72437fb19c0afd.cp312-win_amd64.exp
Gerando c¢digo
Finalizada a geraÆo de c¢digo

In [6]:
print("Algoritmo 1 - Cúbico:", max_subarray_cubic(x_exemplo))


Algoritmo 1 - Cúbico: 187.0


# Algoritmo 2a — Solução Quadrática com Soma Incremental `O(n²)`

## Ideia

A primeira melhoria remove o terceiro laço do algoritmo cúbico.

No algoritmo cúbico, para cada novo `U`, a soma de `x[L:U+1]` era recalculada do zero. Aqui, a soma é reaproveitada:

\[
sum(x[L..U]) = sum(x[L..U-1]) + x[U]
\]

Assim, ao fixar `L`, o algoritmo caminha com `U` para a direita e atualiza a soma em tempo constante.

## Melhoria aplicada sobre o cúbico

A melhoria principal é **salvar estado para evitar recomputação**.

Antes:

```python
for i in range(L, U + 1):
    soma += x[i]
```

Depois:

```python
soma += x[U]
```

O terceiro laço desaparece.

## Linha crítica

A linha crítica passa a ser:

```python
soma += x[U]
```

Ela ainda está dentro de dois laços, mas agora custa `O(1)` para cada par `(L, U)`. Por isso, o algoritmo deixa de ser cúbico e passa a ser quadrático.

## Análise assintótica detalhada

O algoritmo ainda avalia todos os pares `(L, U)`, portanto ainda considera `O(n²)` subvectores.

A diferença é que a soma de cada subvetor é atualizada em tempo constante:

\[
O(n^2) \cdot O(1) = O(n^2)
\]

### Complexidade

- Tempo: `O(n²)`
- Espaço adicional: `O(1)`


In [7]:
%%cython

cpdef double max_subarray_quadratic_incremental(double[:] x):
    cdef Py_ssize_t n = x.shape[0]
    cdef Py_ssize_t L, U
    cdef double max_so_far = 0.0
    cdef double soma

    for L in range(n):
        soma = 0.0
        for U in range(L, n):
            soma += x[U]
            if soma > max_so_far:
                max_so_far = soma

    return max_so_far


Content of stdout:
_cython_magic_669097bf327601a2c6bb6c1263535d5b8ce0beee12439555acd3c74e07ccc54b.c
   Criando biblioteca C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_669097bf327601a2c6bb6c1263535d5b8ce0beee12439555acd3c74e07ccc54b.cp312-win_amd64.lib e objeto C:\Users\Leandro Tosta\.ipython\cython\Users\Leandro Tosta\.ipython\cython\_cython_magic_669097bf327601a2c6bb6c1263535d5b8ce0beee12439555acd3c74e07ccc54b.cp312-win_amd64.exp
Gerando c¢digo
Finalizada a geraÆo de c¢digo

In [8]:
print("Algoritmo 2a - Quadrático incremental:", max_subarray_quadratic_incremental(x_exemplo))


Algoritmo 2a - Quadrático incremental: 187.0


# Algoritmo 2b — Solução Quadrática com Somas Acumuladas `O(n²)`

## Ideia

A segunda melhoria quadrática também remove o terceiro laço, mas usando uma técnica diferente: **pré-processamento por somas acumuladas**.

Criamos um vetor `cum`, no qual:

\[
cum[i] = x[0] + x[1] + \dots + x[i-1]
\]

Assim, a soma de qualquer intervalo `x[L:U+1]` pode ser obtida por:

\[
sum(x[L..U]) = cum[U+1] - cum[L]
\]

## Melhoria aplicada sobre o cúbico

O algoritmo cúbico recalcula a soma percorrendo todos os elementos do intervalo. Aqui, a soma é obtida por duas leituras e uma subtração.

Antes:

```python
for i in range(L, U + 1):
    soma += x[i]
```

Depois:

```python
soma = cum[U + 1] - cum[L]
```

## Linha crítica

A linha crítica é:

```python
soma = cum[U + 1] - cum[L]
```

Ela permite consultar a soma do intervalo em `O(1)`. Porém, como o algoritmo ainda testa todos os pares `(L, U)`, a complexidade continua sendo `O(n²)`.

## Análise assintótica detalhada

O algoritmo possui duas etapas:

1. Construção do vetor acumulado: `O(n)`;
2. teste de todos os pares `(L, U)`: `O(n²)`.

A etapa dominante é a segunda:

\[
O(n) + O(n^2) = O(n^2)
\]

### Complexidade

- Tempo: `O(n²)`
- Espaço adicional: `O(n)`


In [ ]:
%%cython
from libc.stdlib cimport malloc, free

cpdef double max_subarray_quadratic_prefix(double[:] x):
    cdef Py_ssize_t n = x.shape[0]
    cdef Py_ssize_t i, L, U
    cdef double max_so_far = 0.0
    cdef double soma
    cdef double* cum = <double*> malloc((n + 1) * sizeof(double))

    if cum == NULL:
        raise MemoryError()

    try:
        cum[0] = 0.0

        for i in range(n):
            cum[i + 1] = cum[i] + x[i]

        for L in range(n):
            for U in range(L, n):
                soma = cum[U + 1] - cum[L]
                if soma > max_so_far:
                    max_so_far = soma

        return max_so_far

    finally:
        free(cum)


In [ ]:
print("Algoritmo 2b - Quadrático com prefixos:", max_subarray_quadratic_prefix(x_exemplo))


# Algoritmo 3 — Divisão e Conquista `O(n log n)`

## Ideia

A abordagem por divisão e conquista divide o vetor em duas metades:

- metade esquerda;
- metade direita.

A maior soma do vetor completo só pode estar em uma destas três situações:

1. totalmente na metade esquerda;
2. totalmente na metade direita;
3. cruzando o meio do vetor.

O algoritmo resolve recursivamente os casos 1 e 2. O caso 3 é calculado procurando:

- a melhor soma que termina no meio vindo da esquerda;
- a melhor soma que começa depois do meio indo para a direita.

A soma cruzada é a soma dessas duas partes.

## Melhoria aplicada sobre o cúbico

O algoritmo cúbico examina todos os subvectores diretamente. A divisão e conquista reduz o problema em subproblemas menores e calcula o caso de cruzamento em tempo linear por nível de recursão.

A melhoria não remove apenas um laço; ela muda a estratégia:

- de enumeração exaustiva de intervalos;
- para decomposição recursiva do problema.

## Linha crítica

A parte crítica está nos dois laços que calculam a soma cruzada:

```python
for i in range(mid, left - 1, -1)
for i in range(mid + 1, right + 1)
```

Eles são responsáveis pelo custo linear em cada nível da recursão. Sem esse cálculo de cruzamento, o algoritmo ignoraria subvectores que começam em uma metade e terminam na outra, produzindo resultado incorreto.

## Análise assintótica detalhada

A cada chamada, o problema de tamanho `n` é dividido em dois subproblemas de tamanho aproximadamente `n/2`.

Além disso, o cálculo da soma cruzada percorre os elementos do intervalo uma vez, custando `O(n)`.

A recorrência é:

\[
T(n) = 2T(n/2) + O(n)
\]

Pelo Teorema Mestre:

\[
T(n) = O(n \log n)
\]

### Complexidade

- Tempo: `O(n log n)`
- Espaço adicional: `O(log n)` por causa da pilha de recursão.


In [ ]:
%%cython

cdef double _max_subarray_divide_conquer(double[:] x, Py_ssize_t left, Py_ssize_t right):
    cdef Py_ssize_t mid, i
    cdef double soma
    cdef double max_left, max_right, max_crossing
    cdef double max_in_left, max_in_right

    if left > right:
        return 0.0

    if left == right:
        if x[left] > 0.0:
            return x[left]
        return 0.0

    mid = (left + right) // 2

    soma = 0.0
    max_left = 0.0
    for i in range(mid, left - 1, -1):
        soma += x[i]
        if soma > max_left:
            max_left = soma

    soma = 0.0
    max_right = 0.0
    for i in range(mid + 1, right + 1):
        soma += x[i]
        if soma > max_right:
            max_right = soma

    max_crossing = max_left + max_right

    max_in_left = _max_subarray_divide_conquer(x, left, mid)
    max_in_right = _max_subarray_divide_conquer(x, mid + 1, right)

    if max_crossing >= max_in_left and max_crossing >= max_in_right:
        return max_crossing
    elif max_in_left >= max_in_right:
        return max_in_left
    else:
        return max_in_right


cpdef double max_subarray_divide_conquer(double[:] x):
    cdef Py_ssize_t n = x.shape[0]

    if n == 0:
        return 0.0

    return _max_subarray_divide_conquer(x, 0, n - 1)


In [ ]:
print("Algoritmo 3 - Divisão e conquista:", max_subarray_divide_conquer(x_exemplo))


# Algoritmo 4 — Solução Linear `O(n)`

## Ideia

A solução linear percorre o vetor uma única vez.

Ela mantém duas informações:

- `max_ending_here`: melhor soma de um subvetor que termina exatamente na posição atual;
- `max_so_far`: melhor soma encontrada em qualquer posição até agora.

A cada elemento, o algoritmo decide se vale a pena continuar a soma anterior ou recomeçar com o subvetor vazio. Como o subvetor vazio tem soma zero, qualquer soma negativa é descartada.

## Melhoria aplicada sobre o cúbico

O algoritmo cúbico calcula repetidamente todas as somas possíveis. O algoritmo linear usa uma observação mais forte:

> Para decidir a melhor soma terminando na posição atual, basta conhecer a melhor soma que terminava na posição anterior.

Assim, ele não precisa listar todos os pares `(L, U)`. Cada elemento é processado apenas uma vez.

## Linha crítica

A linha crítica é:

```python
max_ending_here = max(0, max_ending_here + x[i])
```

Ela resume a decisão central do algoritmo. Se a soma acumulada ficar negativa, ela é zerada, pois uma soma negativa só prejudicaria qualquer subvetor futuro.

## Invariante principal

Ao final da iteração `i`:

- `max_ending_here` contém a maior soma de um subvetor que termina exatamente em `i`;
- `max_so_far` contém a maior soma de qualquer subvetor dentro de `x[0:i+1]`.

## Análise assintótica detalhada

O algoritmo executa apenas um laço sobre os `n` elementos. Em cada iteração, faz um número constante de operações.

\[
O(n) \cdot O(1) = O(n)
\]

Nenhum algoritmo pode ser assintoticamente melhor do que `O(n)` neste problema, porque é necessário observar todos os elementos do vetor pelo menos uma vez.

### Complexidade

- Tempo: `O(n)`
- Espaço adicional: `O(1)`


In [ ]:
%%cython

cpdef double max_subarray_linear(double[:] x):
    cdef Py_ssize_t n = x.shape[0]
    cdef Py_ssize_t i
    cdef double max_so_far = 0.0
    cdef double max_ending_here = 0.0

    for i in range(n):
        max_ending_here += x[i]

        if max_ending_here < 0.0:
            max_ending_here = 0.0

        if max_ending_here > max_so_far:
            max_so_far = max_ending_here

    return max_so_far


In [ ]:
print("Algoritmo 4 - Linear:", max_subarray_linear(x_exemplo))


# 3. Validação automática dos 5 algoritmos

A célula abaixo executa todos os algoritmos com diferentes vetores pequenos para verificar se as respostas são iguais.


In [ ]:
casos_teste = [
    np.array([31, -41, 59, 26, -53, 58, 97, -93, -23, 84], dtype=np.float64),
    np.array([-5, -2, -9, -1], dtype=np.float64),
    np.array([1, 2, 3, 4], dtype=np.float64),
    np.array([4, -1, 2, 1, -5, 4], dtype=np.float64),
    np.array([], dtype=np.float64),
    np.array([10, -20, 30, -5, 40, -100, 50], dtype=np.float64),
]

funcoes = {
    "Algoritmo 1 - Cúbico": max_subarray_cubic,
    "Algoritmo 2a - Quadrático incremental": max_subarray_quadratic_incremental,
    "Algoritmo 2b - Quadrático prefixos": max_subarray_quadratic_prefix,
    "Algoritmo 3 - Divisão e conquista": max_subarray_divide_conquer,
    "Algoritmo 4 - Linear": max_subarray_linear,
}

for idx, caso in enumerate(casos_teste, start=1):
    respostas = {nome: func(caso) for nome, func in funcoes.items()}
    print(f"\nCaso {idx}: {caso}")
    for nome, resposta in respostas.items():
        print(f"{nome}: {resposta}")

    assert len(set(respostas.values())) == 1, "Os algoritmos retornaram resultados diferentes."

print("\nTodos os testes passaram.")


# 4. Comparação experimental de tempo

A análise assintótica descreve o crescimento teórico. A medição abaixo serve apenas como demonstração prática.

Para evitar que o algoritmo cúbico demore demais, ele é testado com tamanhos menores.


In [ ]:
def medir_tempo(func, x, repeticoes=3):
    tempos = timeit.repeat(lambda: func(x), repeat=repeticoes, number=1)
    return min(tempos)

rng = np.random.default_rng(42)

configuracao = [
    ("Algoritmo 1 - Cúbico", max_subarray_cubic, [20, 40, 80, 120]),
    ("Algoritmo 2a - Quadrático incremental", max_subarray_quadratic_incremental, [100, 300, 600, 1000]),
    ("Algoritmo 2b - Quadrático prefixos", max_subarray_quadratic_prefix, [100, 300, 600, 1000]),
    ("Algoritmo 3 - Divisão e conquista", max_subarray_divide_conquer, [1000, 3000, 6000, 10000]),
    ("Algoritmo 4 - Linear", max_subarray_linear, [1000, 10000, 100000, 500000]),
]

resultados = []

for nome, func, tamanhos in configuracao:
    for n in tamanhos:
        x = rng.normal(loc=0.0, scale=1.0, size=n).astype(np.float64)
        tempo = medir_tempo(func, x)
        resultados.append({"algoritmo": nome, "n": n, "tempo_segundos": tempo})

df_tempos = pd.DataFrame(resultados)
df_tempos


In [ ]:
plt.figure(figsize=(10, 6))

for nome in df_tempos["algoritmo"].unique():
    dados = df_tempos[df_tempos["algoritmo"] == nome]
    plt.plot(dados["n"], dados["tempo_segundos"], marker="o", label=nome)

plt.xlabel("Tamanho do vetor (n)")
plt.ylabel("Tempo mínimo observado (segundos)")
plt.title("Comparação experimental dos algoritmos")
plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.grid(True, which="both")
plt.show()


# 5. Síntese das melhorias

| Algoritmo | Técnica principal | O que melhora em relação ao cúbico | Tempo | Espaço |
|---|---|---|---|---|
| 1 | Enumeração completa | Base de comparação | `O(n³)` | `O(1)` |
| 2a | Soma incremental | Remove o laço interno de recomputação da soma | `O(n²)` | `O(1)` |
| 2b | Somas acumuladas | Pré-processa o vetor para consultar intervalos em `O(1)` | `O(n²)` | `O(n)` |
| 3 | Divisão e conquista | Divide o problema e calcula apenas o cruzamento entre metades | `O(n log n)` | `O(log n)` |
| 4 | Varredura linear | Mantém invariantes suficientes para processar cada elemento uma vez | `O(n)` | `O(1)` |

## Conclusão

A evolução dos algoritmos mostra que a principal fonte de ineficiência do método cúbico é a recomputação de somas já conhecidas. Cada melhoria explora uma estratégia diferente:

- reaproveitar estado;
- pré-processar somas;
- dividir o problema;
- manter invariantes durante uma única varredura.

A solução linear é assintoticamente ótima, pois qualquer algoritmo correto precisa examinar todos os elementos do vetor ao menos uma vez.
